In [5]:


def clean_ayah_text(text, suarh_number, ayah_in_surah):
        
        BASMALA = "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ"
        text = text.replace("\ufeff","")
        
        if ayah_in_surah == 1 and suarh_number not in (1,9):
            
            if text.startswith(BASMALA):
                
                text = text.removeprefix(BASMALA).strip()
        print(text)


clean_ayah_text(
    "بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ الم", 2,1
)

الم


In [ ]:


import json
from normalize import normalize_arabic
from pathlib import Path


folder_name = Path("Database")

file_path = folder_name / "quran.json"

with open (file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
    
for ayah in data:
    normalize= normalize_arabic(ayah["text"])
    ayah["norma_text"] = normalize
    
    
output_path = folder_name / "quran_normalized.json"
with open(output_path, "w", encoding="utf-8") as file:
    json.dump(data, file, indent=4)

In [23]:



import json
from pathlib import Path
from build_normalized import normalize_arabic

file_path = Path("Database") / "quran_normalized.json"

with open(file_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)
    
    
def find_exact(input_user:str, quran_data:str, min_word=3, status_callback=None):
    
    def update_status(msg):
        if status_callback:
            status_callback(msg)
    
    text_user = normalize_arabic(input_user)
    words = text_user.split()
    
    
    if len(words) < min_word:
        print(f"⚠️ Search query is too short. Please enter at least {min_word} words.")
        return []
    
    results = []
    
    for ayah in quran_data:
        
        quran_normalized = ayah["norma_text"]
        
        if text_user in quran_normalized:
            
            match_type = "Exact Match Found" if text_user == quran_normalized else "Partial Match Found"
            
            results.append({
                "Surah Name": ayah["surah_name"],
                "Surah Number": ayah["surah_number"],
                "Ayah Number": ayah["ayah_number"],
                "Ayah": ayah["text"],
                "Match" : match_type
            })
            
            
    if results:
        print(f"\nFound {len(results)} matching result(s) for: '{input_user}'\n")
        for i, res in enumerate(results, 1):
            print("="*55)
            print(f"[{i}] {res['Match']}")
            print("="*55)
            print(f"📖 Surah Name   : {res['Surah Name']} (Number: {res['Surah Number']})")
            print(f"🔢 Ayah Number  : {res['Ayah Number']}")
            print(f"📜 Verse Text   : {res['Ayah']}")
            print("="*55 + "\n")
    else:
        print("\n❌ No matching verses found for your search.")
    
    return results

if __name__ == "__main__":
    user_input = input("Enter Ayah to search: ")
    if isinstance(user_input, str) and user_input.strip():
        find_exact(user_input, dataset)
    else:
        print("Invalid Text Inputs!!")
    
    


Found 1 matching result(s) for: 'انْفِرُوا خِفَافًا وَثِقَالًا وَجَاهِدُوا بِأَمْوَالِكُمْ وَأَنْفُسِكُمْ فِي سَبِيلِ اللَّهِ ۚ ذَٰلِكُمْ خَيْرٌ لَكُمْ إِنْ كُنْتُمْ تَعْلَمُونَ'

[1] Exact Match Found
📖 Surah Name   : سُورَةُ التَّوۡبَةِ (Number: 9)
🔢 Ayah Number  : 41
📜 Verse Text   : انْفِرُوا خِفَافًا وَثِقَالًا وَجَاهِدُوا بِأَمْوَالِكُمْ وَأَنْفُسِكُمْ فِي سَبِيلِ اللَّهِ ۚ ذَٰلِكُمْ خَيْرٌ لَكُمْ إِنْ كُنْتُمْ تَعْلَمُونَ



In [50]:

from pathlib import Path
from rapidfuzz import fuzz, process
import json
from normalize import normalize_arabic
from difflib import SequenceMatcher


file_path = Path("Database") / "quran_normalized.json"

with open(file_path, "r", encoding="utf-8") as f:
    data= json.load(f)
    
user_input = "وَمَا تِلْكَ بيسارك يَا مُوسَىٰ"
quran_normalized = [ayah["norma_text"] for ayah in data]

resaults = process.extract(
    normalize_arabic(user_input),
    quran_normalized,
    
    scorer=fuzz.partial_ratio,limit=3
)



input_words = user_input.split()
# print(resaults)


for ayah in resaults:
    ayah_words = ayah[0]
    
    ayah_words = ayah_words.split()
    matcher = SequenceMatcher(None, input_words, ayah_words)
    for tag, i1,i2, j1,j2 in matcher.get_opcodes():
        
        if tag == "equal":
            print("Match Exactly", input_words[i1:i2])
            
        elif tag == "replace" :
            print("Words Changed with", input_words[i1:i2], ayah_words[j1:j2])
            
        elif  tag == "delete":
            print("Words Appendix", input_words[i1:i2])
            
        elif tag == "insert":
            print("Words Missed from", ayah_words[j1:j2])




# ayah_words = resaults.split()
        

Words Changed with ['وَمَا', 'تِلْكَ', 'بيسارك', 'يَا', 'مُوسَىٰ'] ['يس']
Words Changed with ['وَمَا', 'تِلْكَ', 'بيسارك', 'يَا', 'مُوسَىٰ'] ['وما', 'تلك', 'بيمينك', 'يا', 'موسي']
Words Changed with ['وَمَا', 'تِلْكَ', 'بيسارك', 'يَا', 'مُوسَىٰ'] ['قال', 'اجئتنا', 'لتخرجنا', 'من', 'ارضنا', 'بسحرك', 'يا', 'موسي']


In [ ]:
def find_fuzzy(user_input:str, quran_normalized=quran_normalized, limit=3) -> list:
    
    normalized_input = normalize_arabic(user_input)
    
    user_words= normalized_input.split()
    
    
    matches = process.extract(
        normalized_input, 
        quran_normalized,
        scorer = fuzz.partial_ratio,limit=limit
    )
    
    results= []
    for ayah in matches:
        ayah_words = ayah[0]
        ayah_words = ayah_words.split()
        index = ayah[2]
        score = ayah[1]
        matcher = SequenceMatcher(None, user_words, ayah_words)
        diffs = []
        
        ayah_data= data[index]
        for tag, i1,i2,j1,j2 in matcher.get_opcodes():
            
            if tag == "equal":
                diffs.append({
                    "type":tag
                })
            elif tag in ("replace", "delete"):
                diffs.append({
                    "type":tag,
                    "user_words": user_words[i1:i2],
                    "ayah_words":ayah_words[j1:j2]
                })
                
            elif tag == "insert":
                if 0 < i1 and i2 < len(user_words):
                    diffs.append({
                        "type":tag,
                        "user_words":user_words[i1:i2],
                        "ayah_words":ayah_words[j1:j2]
                    })
                    
        if score == 100 and len(diffs) == 1:
            status = ""
        elif score >= 80 :
            status = ""
        else:
            status = ""
                    
                
        results.append({
            "surah_name": ayah_data["surah_name"],
            "surah_number": ayah_data["surah_number"],
            "ayah_number": ayah_data["ayah_number"],
            "text": ayah_data["text"],
            "text_normalized":ayah_data["norma_text"],
            "score": ayah[1],
            "Differences":diffs ,
            "match_type":status
        })
        
    print(len(diffs))
    print(diffs)
    print(results)
    return results
    
 

if __name__ == "__main__":
    find_fuzzy("يَا أَيُّهَا الَّذِينَ آمَنُوا اتَّقُوا اللَّهَ")


1
[{'type': 'equal'}]
[{'surah_name': 'سُورَةُ البَقَرَةِ', 'surah_number': 2, 'ayah_number': 278, 'text': 'يَا أَيُّهَا الَّذِينَ آمَنُوا اتَّقُوا اللَّهَ وَذَرُوا مَا بَقِيَ مِنَ الرِّبَا إِنْ كُنْتُمْ مُؤْمِنِينَ', 'text_normalized': 'يا ايها الذين امنوا اتقوا الله وذروا ما بقي من الربا ان كنتم مؤمنين', 'score': 100.0, 'Differences': [{'type': 'equal'}]}, {'surah_name': 'سُورَةُ آلِ عِمۡرَانَ', 'surah_number': 3, 'ayah_number': 102, 'text': 'يَا أَيُّهَا الَّذِينَ آمَنُوا اتَّقُوا اللَّهَ حَقَّ تُقَاتِهِ وَلَا تَمُوتُنَّ إِلَّا وَأَنْتُمْ مُسْلِمُونَ', 'text_normalized': 'يا ايها الذين امنوا اتقوا الله حق تقاته ولا تموتن الا وانتم مسلمون', 'score': 100.0, 'Differences': [{'type': 'equal'}]}, {'surah_name': 'سُورَةُ المَائـِدَةِ', 'surah_number': 5, 'ayah_number': 35, 'text': 'يَا أَيُّهَا الَّذِينَ آمَنُوا اتَّقُوا اللَّهَ وَابْتَغُوا إِلَيْهِ الْوَسِيلَةَ وَجَاهِدُوا فِي سَبِيلِهِ لَعَلَّكُمْ تُفْلِحُونَ', 'text_normalized': 'يا ايها الذين امنوا اتقوا الله وابتغوا اليه الوسيله وجا

In [43]:


from pathlib import Path
import json
from rapidfuzz import fuzz, process
from difflib import SequenceMatcher
from normalize import normalize_arabic


def find_fuzzy(user_input: str, quran_data: list, limit=3) -> list:
    """
    دالة للبحث المرن عن أقرب الآيات وإستخراج الاختلافات بين الكلمات.
    """
    # 1. تجهيز المدخلات
    normalized_input = normalize_arabic(user_input)
    user_words = normalized_input.split()
    
    quran_normalized = [ayah["norma_text"] for ayah in quran_data]

    # 2. استخراج أقرب المطابقات
    matches = process.extract(
        normalized_input,
        quran_normalized,
        scorer=fuzz.partial_ratio,
        limit=limit
    )

    results = []

    # 3. معالجة كل نتيجة واستخراج الفروقات
    for matched_text, score, index in matches:
        ayah = quran_data[index]
        matched_words = matched_text.split()

        # مقارنة الكلمات واستخراج الفروقات
        matcher = SequenceMatcher(None, user_words, matched_words)
        diffs = []

        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            if tag != 'equal':
                diffs.append({
                    "type": tag,
                    "user_words": user_words[i1:i2],
                    "ayah_words": matched_words[j1:j2]
                })

        # إضافة النتيجة كـ dictionary مستقل
        results.append({
            "surah_name": ayah["surah_name"],
            "surah_number": ayah["surah_number"],
            "ayah_number": ayah["ayah_number"],
            "text": ayah["text"],
            "text_normalized": ayah["norma_text"],
            "score": score,
            "differences": diffs
        })

    return results

if __name__ == "__main__":
    find_fuzzy

[{'surah_name': 'سُورَةُ يسٓ', 'surah_number': 36, 'ayah_number': 1, 'text': 'يس', 'text_normalized': 'يس', 'score': 100.0, 'differences': [{'type': 'replace', 'user_words': ['وما', 'تلك', 'بيسارك', 'يا', 'موسي'], 'ayah_words': ['يس']}]}]
[{'surah_name': 'سُورَةُ يسٓ', 'surah_number': 36, 'ayah_number': 1, 'text': 'يس', 'text_normalized': 'يس', 'score': 100.0, 'differences': [{'type': 'replace', 'user_words': ['وما', 'تلك', 'بيسارك', 'يا', 'موسي'], 'ayah_words': ['يس']}]}, {'surah_name': 'سُورَةُ طه', 'surah_number': 20, 'ayah_number': 17, 'text': 'وَمَا تِلْكَ بِيَمِينِكَ يَا مُوسَىٰ', 'text_normalized': 'وما تلك بيمينك يا موسي', 'score': 86.36363636363636, 'differences': [{'type': 'replace', 'user_words': ['بيسارك'], 'ayah_words': ['بيمينك']}]}]
[{'surah_name': 'سُورَةُ يسٓ', 'surah_number': 36, 'ayah_number': 1, 'text': 'يس', 'text_normalized': 'يس', 'score': 100.0, 'differences': [{'type': 'replace', 'user_words': ['وما', 'تلك', 'بيسارك', 'يا', 'موسي'], 'ayah_words': ['يس']}]}, {'s